In [1]:
dia_inici_classes = '2026-09-14'
dia_final_classes = '2027-05-07'

festius = [
    ('2026-10-12', '2026-10-12'),  # Tots Sants
    ('2026-12-07', '2026-12-08'),  # Immaculada
    ('2026-12-22', '2027-01-07'),  # Vacances de Nadal
    ('2027-02-08', '2027-02-09'),  # Dia de la Comunitat Valenciana
    ('2027-03-22', '2027-04-29'),  # Vacances de Pasqua
    ('2027-04-29', '2027-05-03'),  # Dia del Treball

    # Setmanes d'examen
    ('2026-11-16', '2026-11-18'),  # Setmana examens 1
    ('2027-02-15', '2027-02-17'),  # Setmana examens 2
    ('2027-05-17', '2027-05-19'),  # Setmana examens 3
]


hores_x_dia_de_la_setmana = {
    # Grup: dl, dt, dc, dj, dv
    'A': (1, 2, 0, 0, 2),
    'B': (1, 0, 2, 2, 0),
}


In [2]:
temes = [
  {
    "codi": "1",
    "tema": "Tema 1",
    "nom": "Introducció a les bases de dades",
    "ra": "RA1",
    "pes": 10
  },
  {
    "codi": "2.1.1",
    "tema": "Tema 2.1",
    "nom": "Creació de taules",
    "ra": "RA2",
    "pes": 13
  },
  {
    "codi": "2.1.2",
    "tema": "Tema 2.1",
    "nom": "Modificació i eliminació de taules",
    "ra": "RA2",
    "pes": 2
  },
  {
    "codi": "2.2.1",
    "tema": "Tema 2.2",
    "nom": "Insercions",
    "ra": "RA4",
    "pes": 3
  },
  {
    "codi": "2.2.2",
    "tema": "Tema 2.2",
    "nom": "Consultes",
    "ra": "RA3",
    "pes": 25
  },
  {
    "codi": "2.2.3",
    "tema": "Tema 2.2",
    "nom": "Actualitzacions",
    "ra": "RA4",
    "pes": 3
  },
  {
    "codi": "2.2.4",
    "tema": "Tema 2.2",
    "nom": "Esborrats",
    "ra": "RA4",
    "pes": 3
  },
  {
    "codi": "2.3",
    "tema": "Tema 2",
    "nom": "Transaccions",
    "ra": "RA4",
    "pes": 10
  },
  {
    "codi": "2.4",
    "tema": "Tema 2",
    "nom": "Procediments emmagatzemats",
    "ra": "RA5",
    "pes": 15
  },
  {
    "codi": "2.5.1",
    "tema": "Tema 2.5",
    "nom": "Índexs",
    "ra": "RA2",
    "pes": 3
  },
  {
    "codi": "2.5.2",
    "tema": "Tema 2.5",
    "nom": "Vistes",
    "ra": "RA2",
    "pes": 2
  },
  {
    "codi": "2.5.3",
    "tema": "Tema 2.5",
    "nom": "Usuaris i privilegis",
    "ra": "RA2",
    "pes": 3
  },
  {
    "codi": "3.1",
    "tema": "Tema 3",
    "nom": "PDM - Model físic de dades",
    "ra": "RA6",
    "pes": 5
  },
  {
    "codi": "3.2",
    "tema": "Tema 3",
    "nom": "M/ER - Model entitat/relació",
    "ra": "RA6",
    "pes": 10
  },
  {
    "codi": "3.3",
    "tema": "Tema 3",
    "nom": "Normalització de models relacionals",
    "ra": "RA6",
    "pes": 5
  },
  {
    "codi": "4",
    "tema": "Tema 4",
    "nom": "Bases de dades no relacionals",
    "ra": "RA7",
    "pes": 13
  }
]


In [3]:
# Programació horària proporcional per al grup A
from datetime import datetime, timedelta
from math import floor

def es_festiu(data, festius):
    """Indica si una data es troba dins d'un període no lectiu."""
    return any(
        datetime.strptime(inici, '%Y-%m-%d').date() <= data <= datetime.strptime(fi, '%Y-%m-%d').date()
        for inici, fi in festius
    )

def obtenir_dies_lectius(data_inici, data_fi):
    """Retorna les dates lectives i les hores disponibles de cada data."""
    dies = []
    data = data_inici
    hores_grup_a = hores_x_dia_de_la_setmana['A']
    while data <= data_fi:
        dia_setmana = data.weekday()
        hores = hores_grup_a[dia_setmana] if dia_setmana < 5 else 0
        if hores > 0 and not es_festiu(data, festius):
            dies.append([data, hores])
        data += timedelta(days=1)
    return dies

def repartir_hores_per_pes(temes, hores_totals):
    """Distribueix totes les hores segons el pes de cada tema."""
    pes_total = sum(tema['pes'] for tema in temes)
    if pes_total <= 0:
        raise ValueError('La suma dels pesos ha de ser superior a zero.')

    quotes = [tema['pes'] / pes_total * hores_totals for tema in temes]
    hores = [floor(quota) for quota in quotes]
    # Les hores restants s'assignen a les quotes amb una part decimal més gran.
    for index in sorted(range(len(temes)), key=lambda i: (quotes[i] - hores[i], -i), reverse=True)[:hores_totals - sum(hores)]:
        hores[index] += 1
    return hores

def planificar_temes(temes, dies_lectius, hores_per_tema):
    """Assigna les hores a cada tema, conservant les hores no consumides del dia."""
    resultats = []
    index_dia = 0

    for tema, hores_tema in zip(temes, hores_per_tema):
        hores_pendents = hores_tema
        data_inici = dies_lectius[index_dia][0] if hores_pendents and index_dia < len(dies_lectius) else None
        data_fi = None

        while hores_pendents and index_dia < len(dies_lectius):
            data, hores_disponibles = dies_lectius[index_dia]
            consum = min(hores_pendents, hores_disponibles)
            hores_pendents -= consum
            dies_lectius[index_dia][1] -= consum
            data_fi = data
            if dies_lectius[index_dia][1] == 0:
                index_dia += 1

        if hores_pendents:
            raise RuntimeError('No hi ha prou hores lectives per planificar tots els temes.')
        resultats.append((tema, hores_tema, data_inici, data_fi))

    return resultats

mesos = ['gener', 'febrer', 'març', 'abril', 'maig', 'juny',
         'juliol', 'agost', 'setembre', 'octubre', 'novembre', 'desembre']

def formatar_data(data):
    return f"{data.day} {mesos[data.month - 1]}"

data_inici = datetime.strptime(dia_inici_classes, '%Y-%m-%d').date()
data_fi = datetime.strptime(dia_final_classes, '%Y-%m-%d').date()
dies_lectius = obtenir_dies_lectius(data_inici, data_fi)
hores_totals = sum(hores for _, hores in dies_lectius)
hores_per_tema = repartir_hores_per_pes(temes, hores_totals)
resultats = planificar_temes(temes, dies_lectius, hores_per_tema)

print('| Codi | Nom del tema | Temporalització |')
print('|------|-------------|-----------------|')
for tema, hores_tema, inici, fi in resultats:
    nom_tema = f"{tema['nom']} ({tema['ra']})"
    temporalitzacio = f"{formatar_data(inici)} - {formatar_data(fi)}"
    print(f"| {tema['codi']} | {nom_tema} | {temporalitzacio} |")

| Codi | Nom del tema | Temporalització |
|------|-------------|-----------------|
| 1 | Introducció a les bases de dades (RA1) | 14 setembre - 25 setembre |
| 2.1.1 | Creació de taules (RA2) | 25 setembre - 13 octubre |
| 2.1.2 | Modificació i eliminació de taules (RA2) | 13 octubre - 16 octubre |
| 2.2.1 | Insercions (RA4) | 16 octubre - 20 octubre |
| 2.2.2 | Consultes (RA3) | 20 octubre - 24 novembre |
| 2.2.3 | Actualitzacions (RA4) | 27 novembre - 30 novembre |
| 2.2.4 | Esborrats (RA4) | 1 desembre - 4 desembre |
| 2.3 | Transaccions (RA4) | 4 desembre - 21 desembre |
| 2.4 | Procediments emmagatzemats (RA5) | 8 gener - 25 gener |
| 2.5.1 | Índexs (RA2) | 26 gener - 29 gener |
| 2.5.2 | Vistes (RA2) | 29 gener - 1 febrer |
| 2.5.3 | Usuaris i privilegis (RA2) | 2 febrer - 5 febrer |
| 3.1 | PDM - Model físic de dades (RA6) | 5 febrer - 19 febrer |
| 3.2 | M/ER - Model entitat/relació (RA6) | 19 febrer - 2 març |
| 3.3 | Normalització de models relacionals (RA6) | 5 març - 9 març